In [ ]:
import os
import sys
import warnings
import time
import json
from natsort import natsorted
from pathlib import PureWindowsPath, PurePosixPath
import pickle
import numpy as np
import xarray as xr
import pandas as pd
import math 

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import seaborn as sns
import cmasher as cmr

from scipy.signal import find_peaks, peak_widths
import scipy.stats as stats
from scipy.stats import skew, median_abs_deviation
import scikit_posthocs as sp
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf

sys.path.append('../utils') 
from utils_tfc import TFC_proto
from utils_tfc import open_minian, xrconcat_recursive, map_ts
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided, add_significance_bar)

#General parameters
dpath_cal_all = r'../../data/11.Post_TFC-20' # The directory for TFC-20
dpath_cal_dcz = r'../../data/10.Post_TFC_DCZ' # The directory for others (TFC-60, TFC-5, TFC-0)
dir_stat = r'../../data/09.Anatomy_behavior'
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] 
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig3'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 1 Cell firing sparness comparison among EC5b, EC3 and CA1 in the novel context exploration

In [ ]:
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

dpath_minian_all = '/Users/yinghao/10.Data/02.Post/11.Post_TFC-20'
test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
test_algori_data = 'post_01_3_overview_QC_spikes_condi_bin_0.2s_new_z'

spike_data = dict()
for i in range(group_size):   
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    

    Sig_bin_baseline_cells = xr.open_dataset(os.path.join(dpath_test, "Spikes_bin_baseline_cells_pool.nc"))  
    Sig_3d = Sig_bin_baseline_cells['Sig_baseline'].expand_dims(dim={"trials": 1}).transpose("unit_id", "trials", "bins").values
    spike_data[group_keys[i]] = Sig_3d

df_sparsity = analyze_sparsity(spike_data, fs=fs)

width_mm = 25  # 
height_mm = 30 #  
epoch_name= 'Explor'

colors = {group_keys[0]: colors_anatomy[0], group_keys[1]:  colors_anatomy[1], group_keys[2]:  colors_anatomy[2]} 
plot_sparsity_violin(df_sparsity, 'Sparseness', width_mm, height_mm, epoch_name, group_keys, colors, dpath_plot, '01_1_Lifetime Sparseness of all cells in the context exploration')
print('All finished************') 

In [ ]:
def plot_sparsity_violin(df, key_stat, width_mm, height_mm, epoch_name, region_order, colors, output_path, title):
    """
    Generates a high-density violin + scatter plot mathematically locked to exact millimeter dimensions,
    and logs comprehensive cell counts, medians, and Kruskal-Wallis/Dunn stats to a JSON.
    Assumes a single condition/epoch, plotting only by 'Region'.
    """
    set_pub_style()      
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    
    metadata = {
        "Figure_Title": title, 
        "Statistics": {
            "Group_Data": {},
            "Kruskal_Wallis": None,
            "Dunns_Posthoc": None
        }
    }
    
    # --- 1. PRE-FLIGHT Y-AXIS SCALING ---
    y_min_data = df[key_stat].min()
    y_max_data = df[key_stat].max()        
    # Bottom buffer: 0.05. Top buffer: 20% headroom for double brackets.
    ax.set_ylim(max(0.0, y_min_data - 0.05), y_max_data * 1.2)
    
    # --- 2. DRAW DISTRIBUTIONS (Violin + Strip) ---
    sns.violinplot(
        data=df, x='Region', y=key_stat, 
        hue='Region', order=region_order, palette=colors,
        cut=0, inner=None, linewidth=0.5, ax=ax, dodge=False, legend=False)
    
    for collection in ax.collections: 
        collection.set_alpha(0.4)         
    sns.stripplot(
        data=df, x='Region', y=key_stat,
        hue='Region', order=region_order, palette=colors,
        dodge=False, alpha=0.3, size=1.0, jitter=0.15, ax=ax, zorder=1, legend=False)
    

    # --- 3. MEDIANS & STATISTICAL TESTING ---
    # Mapping directly to the x-axis index, offsets are just integer positions
    x_positions = {region: i for i, region in enumerate(region_order)}
    
    # A. Calculate and Draw Medians
    for region in region_order:
        subset = df[df['Region'] == region]
        if len(subset) > 0:
            n_cells = len(subset)
            med_val = subset[key_stat].median()                
            
            # Log Data to JSON
            metadata["Statistics"]["Group_Data"][region] = {
                "N_cells": n_cells,
                "Median": float(med_val)
            }
            
            x_pos = x_positions[region]                
            ax.scatter(x_pos, med_val, marker='D', color='white', 
                       edgecolor='black', s=4, zorder=5, linewidth=0.5)
                           
    # B. Automated Stats
    vals_grp0 = df[df['Region'] == region_order[0]][key_stat]
    vals_grp1 = df[df['Region'] == region_order[1]][key_stat]
    vals_grp2 = df[df['Region'] == region_order[2]][key_stat]      
    
    # Ensure data in all groups before running Kruskal
    if len(vals_grp0) > 2 and len(vals_grp1) > 2 and len(vals_grp2) > 2:
        stat, p_kw = stats.kruskal(vals_grp0, vals_grp1, vals_grp2)                
        
        # Log Kruskal-Wallis omnibus results
        metadata["Statistics"]["Kruskal_Wallis"] = {
            "H_statistic": float(stat),
            "p_value": float(p_kw)
        }            
        
        if p_kw < 0.05:
            # Dunn's Post-Hoc Test
            p_mat = sp.posthoc_dunn(df, val_col=key_stat, group_col='Region', p_adjust='bonferroni')                                         
            
            # Compare Group 0 vs Group 1, and Group 0 vs Group 2
            p_0_vs_1 = p_mat.loc[region_order[0], region_order[1]]
            p_0_vs_2 = p_mat.loc[region_order[0], region_order[2]]            
            
            # Log Post-Hoc results to JSON for caption writing
            metadata["Statistics"]["Dunns_Posthoc"] = {
                f"{region_order[0]}_vs_{region_order[1]}": float(p_0_vs_1),
                f"{region_order[0]}_vs_{region_order[2]}": float(p_0_vs_2)
            }
            
            # Find local roof above the highest point in the entire dataset
            roof = df[key_stat].max()                                
            
            # Bracket 1 (Inner comparison: Grp 0 vs Grp 1)
            if p_0_vs_1 < 0.05:
                roof = add_significance_bar(ax, x1=x_positions[region_order[1]], x2=x_positions[region_order[0]], 
                    y_max=roof, text=get_asterisks(p_0_vs_1))                                                
                    
            # Bracket 2 (Outer comparison: Grp 0 vs Grp 2)
            if p_0_vs_2 < 0.05:
                add_significance_bar(ax, x1=x_positions[region_order[2]], x2=x_positions[region_order[0]], 
                    y_max=roof, text=get_asterisks(p_0_vs_2))

    # --- 4. AESTHETICS & EXPORT ---
    ax.set_ylabel('Lifetime sparseness', labelpad=0.5)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    ax.set_xlabel('')       
    ax.tick_params(axis='x', labelsize=6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Strict layout padding
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()        

    save_metadata_json(metadata, output_path, title)


def calculate_treves_rolls_sparseness(spike_vector):
    """
    Calculates Lifetime Sparseness (1 - a) for a 1D array of spikes.
    Values near 1 = Sparse. Values near 0 = Dense.
    """
    if np.sum(spike_vector) == 0:
        return np.nan # Undefined for completely dead cells       
    N = len(spike_vector)
    mean_sq = (np.sum(spike_vector) / N) ** 2
    mean_of_sqs = np.sum(spike_vector ** 2) / N    
    # Add epsilon to prevent division by zero
    a = mean_sq / (mean_of_sqs + 1e-9)
    return 1.0 - a

def analyze_sparsity(spikes_dict, fs=5.0):
    """
    Extracts sparsity metrics for all brain regions.   
    Parameters:
    -----------
    spikes_dict : dict {'EC5b': array(cells, trials, bins), 'CA1': ...}
    epochs : dict {'Tone': (15, 115), 'Shock': (215, 230)}
    fs : float Sampling rate in Hz (5.0 for 0.2s bins).
    """
    records = []    
    for region, data_3d in spikes_dict.items():
        n_cells, n_trials, n_bins = data_3d.shape       
        # Flatten trials and bins to get the "Lifetime" vector for each cell
        # Shape: (n_cells, total_epoch_bins)
        lifetime_data = data_3d.reshape(n_cells, -1)            
        for i in range(n_cells):
            cell_trace = lifetime_data[i, :]              
            # 1. Mean Event Rate (Hz)
            # Sum of spikes / Total time in seconds
            total_time_sec = len(cell_trace) / fs
            mean_rate = np.sum(cell_trace) / total_time_sec                
            # 2. Zero-Inflation (Fraction of silent bins)
            fraction_silent = np.sum(cell_trace == 0) / len(cell_trace)               
            # 3. Treves-Rolls Sparseness
            sparseness = calculate_treves_rolls_sparseness(cell_trace)               
            # Only record cells that fired at least once in the session 
            # (to avoid comparing dead cells)
            if not np.isnan(sparseness):
                records.append({
                    'Region': region,
                    'MeanRate_Hz': mean_rate,
                    'FractionSilent': fraction_silent,
                    'Sparseness': sparseness})                    
    return pd.DataFrame(records)

## 2 DCZ session of EC3 cells--statitical analysis of cell activity between pre and post-DCZ injection

In [ ]:
def extract_paired_data_among_group(df, para_key1, para_key2, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]][[para_key1, para_key2]].to_numpy())
    return data_group
def extract_data_among_group(df, para_keys, group_keys, group_size):
    data_paras =[]
    for para_key in para_keys:
        data_group = []
        for i in range(group_size):
            data_group.append(df.loc[group_keys[i]][para_key].values)
        data_paras.append(data_group)
    return data_paras

group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_10_2_event_rate_QC_spikes' 

ds_all_event_rate = []
ds_all_group_sta =[]
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_dcz, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "DCZ_event_rate_with_spikes.csv"))  
    ds_all_event_rate.append(df_event_rate)
    
    df_sta = pd.read_csv(os.path.join(dpath_test, group_name[i]+"_statistics_with_spikes.csv"))         
    ds_all_group_sta.append(df_sta)
df_all = pd.concat(ds_all_group_sta, keys=group_keys ,names=["group", "row"])

#PLot
height_mm = 30 #  

 # 01. Over all Event rate plot ---pre and post DCZ
width_mm = 30  # 
colors = ['#999999', 'black']  #Pre-DCZ (Neutral Gray) vs Post-DCZ (Active Blue)
#colors =colors_beh_i
list_data = extract_paired_data_among_group(df_all, 'Event_rate_pre_DCZ', 'Event_rate_post_DCZ', group_keys, group_size)
plot_stat_paired_multigroup(list_data, group_keys,'Event rate (Hz)' , ['Pre_DCZ', 'Post_DCZ'], 1, width_mm, height_mm, colors, dpath_plot, '02_1_Pre_and_Post_DCZ_event_rate_mean_comparison')

# 02. scattering plot
palette = {'Excited': '#0A7373', 'Inhibited': '#A31662', 'Stable': '#e0e0e0'}
width_mm = 30  # 
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_dcz, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "DCZ_event_rate_with_spikes.csv"))     
    plot_scattering_cell_shift(df_event_rate, width_mm, height_mm, palette, dpath_plot, '02_2_Pre and Post DCZ event rate-scattering_' + group_keys[i])

# 03. scattering Event Rate MI plot
width_mm = 35 # 
list_data_paras = extract_data_among_group(df_all, ['Excited_MI', 'Inhibited_MI'], group_keys, group_size)
para_labels = ['Facilitated\ncells', "Suppressed\ncells"]
colors = colors_beh_i
plot_stat_unpaired_multigroup(list_data_paras, para_labels, group_keys, 'Event rate MI', width_mm, height_mm, colors, dpath_plot, '02_3_Pre_and_Post_DCZ event rate-MI comparison')

# 04. scattering statisticls
width_mm = 30  # 
colors = ['#0A7373', '#A31662']  #Excited vs Inhibited
list_data = extract_paired_data_among_group(df_all, 'Excited_proportion', 'Inhibited_proportion', group_keys, group_size)
list_data = [data*100 for data in list_data]
plot_stat_paired_multigroup(list_data, group_keys,'Proportion (%)', ['Facilitated', 'Suppressed'], 1, width_mm, height_mm, colors,  dpath_plot, 
                            'sup_02_1_Pre_and_Post_DCZ event rate-scattering_proportion', flag_legend=True)

print('All finished************') 

In [ ]:
def plot_stat_paired_multigroup(data_list, group_labels, y_label, bar_labels, flag_pair, width_mm, height_mm, colors, output_path, title, flag_legend=False):
    """
    Plots paired data for 2 or 3 groups using pure Matplotlib for exact spatial alignment.
    30-50mm width, 30mm height. Includes minimal square legend for bar_labels and p-value metadata.
    """
    set_pub_style() 
    
    n_groups = len(data_list)
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Statistics": {}}
    edge_gray = (0, 0, 0, 0.8)  # Equivalent to #333333 (80% black)
    conn_gray = (0, 0, 0, 0.3)  # Equivalent to ~#b3b3b3 (30% black)
    
    offset = 0.2
    box_width = 0.25
    
    # --- 2. PRE-CALCULATE Y-AXIS LIMITS ---
    global_max = max([np.nanmax(g) for g in data_list])
    axis_range = global_max * 1.15
    ax.set_ylim(0, axis_range) # 35% headroom for brackets
    
    # --- 3. LOOP THROUGH GROUPS ---
    for i, group_data in enumerate(data_list):
        valid_idx = ~np.isnan(group_data).any(axis=1)
        clean_data = group_data[valid_idx]
        
        pre_data = clean_data[:, 0]
        post_data = clean_data[:, 1]
        n_pairs = len(pre_data)
        
        metadata["Statistics"][group_labels[i]] = {"N_pairs": n_pairs}
        if n_pairs == 0: continue
        
        local_max = np.max(clean_data)        
        pos_pre = i - offset
        pos_post = i + offset        
        
        # A. Draw Connecting Lines
        for j in range(n_pairs):
            ax.plot([pos_pre, pos_post], [pre_data[j], post_data[j]], color=conn_gray, lw=0.3, zorder=2)
            
        # B. Box Plots
        # Base kwargs shared by both
        base_kwargs = dict(patch_artist=True, showfliers=False, zorder=1,
                           whiskerprops=dict(color=edge_gray, linewidth=0.5),
                           capprops=dict(color=edge_gray, linewidth=0.5))        

        # PRE-BOX (Gray Box -> Black Median)
        ax.boxplot(pre_data, positions=[pos_pre], widths=box_width, 
                   boxprops=dict(facecolor=colors[0], edgecolor=edge_gray, linewidth=0.5), 
                   medianprops=dict(color='black', linewidth=0.75), **base_kwargs) 
                   
        # POST-BOX (Black Box -> White Median)
        if colors[1]== 'black':
            median_color = 'white'
        else:
            median_color = 'black'
        ax.boxplot(post_data, positions=[pos_post], widths=box_width, 
                   boxprops=dict(facecolor=colors[1], edgecolor=edge_gray, linewidth=0.5), 
                   medianprops=dict(color=median_color, linewidth=0.75), **base_kwargs)

        # C. Scatter Points
        ax.scatter(np.full_like(pre_data, pos_pre), pre_data, s=1.5, 
                   color=colors[0], edgecolors='white', linewidth=0.25, zorder=2) 
        ax.scatter(np.full_like(post_data, pos_post), post_data, s=1.5, 
                   color=colors[1], edgecolors='white', linewidth=0.25, zorder=2)   
                   
        # D. Automated Statistics (And logging exact P-values to JSON)
        p_val_exact = None
        add_stat_annotation_two_sided(ax, pre_data, post_data, pos_pre, pos_post, local_max, ttest=0, paired=flag_pair)
        # Re-calculate to extract exact p-value for metadata
        if flag_pair == 1:
            _, p_val_exact = stats.wilcoxon(pre_data, post_data)
        else:
            _, p_val_exact = stats.mannwhitneyu(pre_data, post_data, alternative='two-sided')

        if p_val_exact is not None:
            metadata["Statistics"][group_labels[i]]["P_Value"] = float(p_val_exact)
            metadata["Statistics"][group_labels[i]]["Mean_pre"] = float(np.mean(pre_data))
            metadata["Statistics"][group_labels[i]]["Mean_post"] = float(np.mean(post_data))
        
    # --- 4. FORMATTING, LEGEND, & AESTHETICS ---
    ax.set_xticks(range(n_groups))
    ax.set_xticklabels(group_labels, fontsize=7)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # ADD OPTIMAL LEGEND (Square markers represent box plots)
    custom_patches = [Patch(facecolor=colors[i], edgecolor=edge_gray, linewidth=0.5, label=bar_labels[i]) for i in range(2)]

    if flag_legend:
        ax.legend(handles=custom_patches, 
                  frameon=False, 
                  loc='upper left', 
                  fontsize=4.5,
                  handlelength=1.0, 
                  handletextpad=0.3,
                  labelspacing=0.2)

    # --- 6. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_scattering_cell_shift(df, width_mm, height_mm, palette, output_path, title):
    """
    Plots a scatter plot of Pre vs Post event rates.
    Highlights Excited and Inhibited cells with independent color palettes.
    Locked to a square mm layout
    """
    set_pub_style()
    # Enforce perfectly square geometry
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Data_Summary": {}, "Statistics": {}}

    # --- 1. CATEGORIZATION & METADATA ---
    threshold_up = 1.3   
    threshold_down = 0.7 
    
    df['Category'] = 'Stable'
    df.loc[df['Post'] > df['Pre'] * threshold_up, 'Category'] = 'Excited'
    df.loc[df['Post'] < df['Pre'] * threshold_down, 'Category'] = 'Inhibited'
    
    counts = df['Category'].value_counts()
    total_cells = len(df)
    props = (counts / total_cells * 100).round(1)
    
    metadata["Data_Summary"] = {
        "Total_Cells": int(total_cells),
        "Excited": {"Count": int(counts.get('Excited', 0)), "Percent": float(props.get('Excited', 0))},
        "Inhibited": {"Count": int(counts.get('Inhibited', 0)), "Percent": float(props.get('Inhibited', 0))},
        "Stable": {"Count": int(counts.get('Stable', 0)), "Percent": float(props.get('Stable', 0))}}
    if total_cells > 2:
        stat_w, p_w = stats.wilcoxon(df['Pre'], df['Post'])
        metadata["Statistics"]["Wilcoxon_Paired"] = {"Statistic": float(stat_w), "p_value": float(p_w)}

    # --- 2. VISUAL OPTIMIZATION TRICKS ---
    
    df_stable = df[df['Category'] == 'Stable']
    df_inhib = df[df['Category'] == 'Inhibited']
    df_excit = df[df['Category'] == 'Excited']

    # Determine dynamic plot limits based on data maximums
    data_max = max(df['Pre'].max(), df['Post'].max())
    limit = data_max * 1.05 
    
    # --- 3. PLOTTING (Micro-scaled for 30mm) ---
    # s=1.0 keeps dots crisp at 30mm
    ax.scatter(df_stable['Pre'], df_stable['Post'], 
               c=palette['Stable'], s=1.0, alpha=1, linewidth=0, zorder=1, rasterized=True)
               
    ax.scatter(df_inhib['Pre'], df_inhib['Post'], 
               c=palette['Inhibited'], s=1.0, alpha=0.85, linewidth=0, zorder=2, rasterized=True)
               
    ax.scatter(df_excit['Pre'], df_excit['Post'], 
               c=palette['Excited'], s=1.0, alpha=0.85, linewidth=0, zorder=3, rasterized=True)

    # --- 4. THRESHOLD LINES (Micro-scaled) ---
    ax.plot([0, limit], [0, limit], color='#333333', linestyle='--', lw=0.5, zorder=0)
    # Cones drop to hair-thin 0.3
    ax.plot([0, limit], [0, limit * threshold_up], color='gray', linestyle=':', lw=0.3, zorder=0)
    ax.plot([0, limit], [0, limit * threshold_down], color='gray', linestyle=':', lw=0.3, zorder=0)

    # --- 5. MINIMALIST TEXT ANNOTATION ---
    # Fontsize reduced to 5 to avoid crushing the data points
    ax.text(0.05, 0.95, f"{props.get('Excited', 0)}%", 
            transform=ax.transAxes, color=palette['Excited'], 
            ha='left', va='top', fontsize=6)
            
    ax.text(0.95, 0.05, f"{props.get('Inhibited', 0)}%", 
            transform=ax.transAxes, color=palette['Inhibited'], 
            ha='right', va='bottom', fontsize=6)

    # --- 6. FORMATTING ---
    ax.set_aspect('equal') 
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    
    ax.set_xlabel('Pre-DCZ rate (Hz)',  labelpad=1)
    ax.set_ylabel('Post-DCZ rate (Hz)', labelpad=1)
    
    # Restrict to maximum 3 or 4 ticks so numbers don't overlap on a 30mm axis
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))  
    # Tick length stays at 2pt
    #ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 7. STRICT EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)      
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_stat_unpaired_multigroup(data_list, para_labels, group_labels, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots unpaired data (Modulation Index) across multiple parameters and animal groups.
    Utilizes a Box + Strip plot layout, ideal for small sample sizes 
    """
    set_pub_style()  
    n_params = len(data_list)
    n_hues = len(data_list[0])  # Number of animal groups (e.g., 2 or 3)
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    edge_gray = (0, 0, 0, 0.8)  # 80% black for clean edges
    
    # --- 1. LAYOUT MATH ---
    # Determine dodge offsets dynamically based on whether there are 2 or 3 animal groups
    if n_hues == 2:
        offsets = [-0.2, 0.2]
        box_width = 0.25
    elif n_hues == 3:
        offsets = [-0.28, 0.0, 0.28]
        box_width = 0.2
    else:
        raise ValueError("This function is optimized for 2 or 3 hue groups.")

    # --- 2. DYNAMIC Y-AXIS SCALING ---
    # Safely extract all values to find absolute global bounds
    all_vals = [val for param in data_list for hue in param for val in hue if not np.isnan(val)]
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min
    
    # 25% headroom for statistical brackets
    ax.set_ylim(global_min - (y_range * 0.05), global_max + (y_range * 0.35))
    # Draw the crucial zero-baseline for Modulation Index
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.75, zorder=0, alpha=0.5)

    # --- 3. LOOP THROUGH PARAMETERS (e.g., Facilitated vs Suppressed) ---
    for p_idx, param_data in enumerate(data_list):
        metadata["Statistics"][para_labels[p_idx]] = {}
        
        # Local ceiling for bracket stacking
        local_max = max([np.nanmax(h) if len(h) > 0 else -np.inf for h in param_data])
        current_roof = local_max + (y_range * 0.08)
        
        # A. Plot Each Animal Group (Hue)
        for h_idx, hue_data in enumerate(param_data):
            clean_data = hue_data[~np.isnan(hue_data)]
            n_animals = len(clean_data)
            
            metadata["Statistics"][para_labels[p_idx]][group_labels[h_idx]] = {"N_animals": n_animals,
                                                                              'Mean values': np.mean(clean_data)}
            if n_animals == 0: continue
            
            x_pos = p_idx + offsets[h_idx]
            
            # Boxplot
            color = colors[h_idx]
            # Face is transparent (0.4), edges and whiskers are solid (1.0)
            face_color_rgba = mcolors.to_rgba(color, alpha=0.4)               
            # Pure Matplotlib Boxplot
            ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))               
            # Pure Matplotlib Scatter (s=2.5
            jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)            
            #x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=n_mice)
            ax.scatter(jittered_x, clean_data, s=2.5, color=color, alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # B. Unpaired Statistical Brackets (Comparisons against Group 0)
        # Check Group 0 vs Group 1
        if len(param_data[0]) > 2 and len(param_data[1]) > 2:
            # use ttest=0 (Mann-Whitney) and paired=0 for independent animals
            current_roof = add_stat_annotation_two_sided(
                ax, param_data[0], param_data[1], 
                p_idx + offsets[0], p_idx + offsets[1], 
                current_roof, ttest=0, paired=0)
            
            # Log exact raw p-value to metadata
            _, p_mwu = stats.mannwhitneyu(param_data[0], param_data[1], alternative='two-sided')
            metadata["Statistics"][para_labels[p_idx]][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)

        # Check Group 0 vs Group 2 (If 3 groups exist)
        if n_hues == 3 and len(param_data[0]) > 2 and len(param_data[2]) > 2:
            # Stack the bracket higher using the returned roof + a small margin
            current_roof += (y_range * 0.08)
            add_stat_annotation_two_sided(
                ax, param_data[0], param_data[2], 
                p_idx + offsets[0], p_idx + offsets[2], 
                current_roof, ttest=0, paired=0)
            
            # Log exact raw p-value to metadata
            # NOTE: If evaluating 3 groups for final publication, a Kruskal-Wallis omnibus test 
            # with Bonferroni correction is statistically mandated before Mann-Whitney.
            _, p_mwu = stats.mannwhitneyu(param_data[0], param_data[2], alternative='two-sided')
            metadata["Statistics"][para_labels[p_idx]][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

    # --- 4. FORMATTING & AESTHETICS ---
    ax.set_xticks(range(n_params))
    ax.set_xticklabels(para_labels, fontsize=6)
    ax.set_ylabel(y_label, labelpad=1)
    
    # 5 ticks max handles symmetric negative/positive scaling well
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    #ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    # --- 6. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 3 DCZ session of CA1 cells--statitical analysis of cell activity between pre and post-DCZ injection

In [ ]:
#  For CA1
def extract_paired_data_among_group(df, para_key1, para_key2, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]][[para_key1, para_key2]].to_numpy())
    return data_group
def extract_data_among_group(df, para_keys, group_keys, group_size):
    data_paras =[]
    for para_key in para_keys:
        data_group = []
        for i in range(group_size):
            data_group.append(df.loc[group_keys[i]][para_key].values)
        data_paras.append(data_group)
    return data_paras

group_name = ['02.CA1-C', '03.CA1-I'] 
group_keys = ['CA1-C', 'CA1-I']
group_size = len(group_name)

test_algori_data = 'post_10_2_event_rate_QC_spikes' 

ds_all_event_rate = []
ds_all_group_sta =[]
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_dcz, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "DCZ_event_rate_with_spikes.csv"))  
    ds_all_event_rate.append(df_event_rate)
    
    df_sta = pd.read_csv(os.path.join(dpath_test, group_name[i]+"_statistics_with_spikes.csv"))         
    ds_all_group_sta.append(df_sta)
df_all = pd.concat(ds_all_group_sta, keys=group_keys ,names=["group", "row"])

#PLot
height_mm = 30 #  

# 01. Over all Event rate plot ---pre and post DCZ
width_mm = 30  # 
colors = ['#999999', 'black']
list_data = extract_paired_data_among_group(df_all, 'Event_rate_pre_DCZ', 'Event_rate_post_DCZ', group_keys, group_size)
plot_stat_paired_multigroup(list_data, group_keys,'Event rate (Hz)' , ['Pre_DCZ', 'Post_DCZ'], 1, width_mm, height_mm, colors, dpath_plot, '03_1_Pre_and_Post_DCZ_event_rate_mean_comparison-CA1')

# 02. scattering plot
width_mm = 30  # 
palette = {'Excited': '#0A7373', 'Inhibited': '#A31662', 'Stable': '#e0e0e0'}
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_dcz, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "DCZ_event_rate_with_spikes.csv"))     
    plot_scattering_cell_shift(df_event_rate, width_mm, height_mm, palette, dpath_plot, f'03_2_Pre and Post DCZ event rate-scattering_CA1_{str(i)}_{group_keys[i]}')

# 03. scattering Event Rate MI plot
width_mm = 40 # 
list_data_paras = extract_data_among_group(df_all, ['Excited_MI', 'Inhibited_MI'], group_keys, group_size)
para_labels = ['Facilitated\ncells', "Suppressed\ncells"]
plot_stat_unpaired_multigroup(list_data_paras, para_labels, group_keys, 'Event rate MI', width_mm, height_mm, colors_beh_i, dpath_plot, '03_3_Pre_and_Post_DCZ event rate-MI comparison-CA1')

# 04. scattering statisticls
width_mm = 30  # 
colors = ['#0A7373', '#A31662']  #Facilited vs suppressed
list_data = extract_paired_data_among_group(df_all, 'Excited_proportion', 'Inhibited_proportion', group_keys, group_size)
list_data = [data*100 for data in list_data]
plot_stat_paired_multigroup(list_data, group_keys,'Proportion (%)', ['Facilitated', 'Suppressed'], 1, width_mm, height_mm, colors,  dpath_plot, '03_4_Pre_and_Post_DCZ event rate-scattering_proportion-CA1')
print('All finished************') 

In [ ]:
def plot_stat_paired_multigroup(data_list, group_labels, y_label, bar_labels, flag_pair, width_mm, height_mm, colors, output_path, title, flag_tail=2, flag_legend=False):
    """
    Plots paired data for 2 or 3 groups using pure Matplotlib for exact spatial alignment.
    30-50mm width, 30mm height. Includes minimal square legend for bar_labels and p-value metadata.
    """
    set_pub_style() 
    
    n_groups = len(data_list)
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Statistics": {}}
    edge_gray = (0, 0, 0, 0.8)  # Equivalent to #333333 (80% black)
    conn_gray = (0, 0, 0, 0.3)  # Equivalent to ~#b3b3b3 (30% black)
    
    offset = 0.2
    box_width = 0.25
    
    # --- 2. PRE-CALCULATE Y-AXIS LIMITS ---
    global_max = max([np.nanmax(g) for g in data_list])
    axis_range = global_max * 1.15
    ax.set_ylim(0, axis_range) # 35% headroom for brackets
    
    # --- 3. LOOP THROUGH GROUPS ---
    for i, group_data in enumerate(data_list):
        valid_idx = ~np.isnan(group_data).any(axis=1)
        clean_data = group_data[valid_idx]
        
        pre_data = clean_data[:, 0]
        post_data = clean_data[:, 1]
        n_pairs = len(pre_data)
        
        metadata["Statistics"][group_labels[i]] = {"N_pairs": n_pairs}
        if n_pairs == 0: continue
        
        local_max = np.max(clean_data)        
        pos_pre = i - offset
        pos_post = i + offset        
        
        # A. Draw Connecting Lines
        for j in range(n_pairs):
            ax.plot([pos_pre, pos_post], [pre_data[j], post_data[j]], color=conn_gray, lw=0.3, zorder=2)
            
        # B. Box Plots
        # Base kwargs shared by both
        base_kwargs = dict(patch_artist=True, showfliers=False, zorder=1,
                           whiskerprops=dict(color=edge_gray, linewidth=0.5),
                           capprops=dict(color=edge_gray, linewidth=0.5))        

        # PRE-BOX (Gray Box -> Black Median)
        ax.boxplot(pre_data, positions=[pos_pre], widths=box_width, 
                   boxprops=dict(facecolor=colors[0], edgecolor=edge_gray, linewidth=0.5), 
                   medianprops=dict(color='black', linewidth=0.75), **base_kwargs) 
                   
        # POST-BOX (Black Box -> White Median)
        if colors[1]== 'black':
            median_color = 'white'
        else:
            median_color = 'black'
        ax.boxplot(post_data, positions=[pos_post], widths=box_width, 
                   boxprops=dict(facecolor=colors[1], edgecolor=edge_gray, linewidth=0.5), 
                   medianprops=dict(color=median_color, linewidth=0.75), **base_kwargs)

        # C. Scatter Points
        ax.scatter(np.full_like(pre_data, pos_pre), pre_data, s=1.5, 
                   color=colors[0], edgecolors='white', linewidth=0.25, zorder=2) 
        ax.scatter(np.full_like(post_data, pos_post), post_data, s=1.5, 
                   color=colors[1], edgecolors='white', linewidth=0.25, zorder=2)   
                   
        # D. Automated Statistics (And logging exact P-values to JSON)
        p_val_exact = None
        if flag_tail == 2:
            add_stat_annotation_two_sided(ax, pre_data, post_data, pos_pre, pos_post, local_max, ttest=0, paired=flag_pair)
            # Re-calculate to extract exact p-value for metadata
            if flag_pair == 1:
                _, p_val_exact = stats.wilcoxon(pre_data, post_data)
            else:
                _, p_val_exact = stats.mannwhitneyu(pre_data, post_data, alternative='two-sided')
                
        if flag_tail == 1:
            if flag_pair == 1:
                stat, p_val_exact = stats.wilcoxon(pre_data, post_data, alternative='less')
            elif flag_pair == 0:
                stat, p_val_exact = stats.mannwhitneyu(pre_data, post_data, alternative='less')
            star = get_asterisks(p_val_exact)
            add_significance_bar(ax, pos_pre, pos_post, local_max, star, color='black')
            
        if p_val_exact is not None:
            metadata["Statistics"][group_labels[i]]["P_Value"] = float(p_val_exact)
            metadata["Statistics"][group_labels[i]]["Mean_pre"] = float(np.mean(pre_data))
            metadata["Statistics"][group_labels[i]]["Mean_post"] = float(np.mean(post_data))
        
    # --- 4. FORMATTING, LEGEND, & AESTHETICS ---
    ax.set_xticks(range(n_groups))
    ax.set_xticklabels(group_labels, fontsize=7)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # ADD OPTIMAL LEGEND (Square markers represent box plots)
    custom_patches = [Patch(facecolor=colors[i], edgecolor=edge_gray, linewidth=0.5, label=bar_labels[i]) for i in range(2)]

    if flag_legend:
        ax.legend(handles=custom_patches, 
                  frameon=False, 
                  loc='upper left', 
                  fontsize=4.5,
                  handlelength=1.0, 
                  handletextpad=0.3,
                  labelspacing=0.2)

    # --- 6. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_scattering_cell_shift(df, width_mm, height_mm, palette, output_path, title):
    """
    Plots a scatter plot of Pre vs Post event rates.
    Highlights Excited and Inhibited cells with independent color palettes.
    Locked to a square mm layout
    """
    set_pub_style()
    # Enforce perfectly square geometry
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Data_Summary": {}, "Statistics": {}}

    # --- 1. CATEGORIZATION & METADATA ---
    threshold_up = 1.3   
    threshold_down = 0.7 
    
    df['Category'] = 'Stable'
    df.loc[df['Post'] > df['Pre'] * threshold_up, 'Category'] = 'Excited'
    df.loc[df['Post'] < df['Pre'] * threshold_down, 'Category'] = 'Inhibited'
    
    counts = df['Category'].value_counts()
    total_cells = len(df)
    props = (counts / total_cells * 100).round(1)
    
    metadata["Data_Summary"] = {
        "Total_Cells": int(total_cells),
        "Excited": {"Count": int(counts.get('Excited', 0)), "Percent": float(props.get('Excited', 0))},
        "Inhibited": {"Count": int(counts.get('Inhibited', 0)), "Percent": float(props.get('Inhibited', 0))},
        "Stable": {"Count": int(counts.get('Stable', 0)), "Percent": float(props.get('Stable', 0))}}
    if total_cells > 2:
        stat_w, p_w = stats.wilcoxon(df['Pre'], df['Post'])
        metadata["Statistics"]["Wilcoxon_Paired"] = {"Statistic": float(stat_w), "p_value": float(p_w)}

    # --- 2. VISUAL OPTIMIZATION TRICKS ---
    
    df_stable = df[df['Category'] == 'Stable']
    df_inhib = df[df['Category'] == 'Inhibited']
    df_excit = df[df['Category'] == 'Excited']

    # Determine dynamic plot limits based on data maximums
    data_max = max(df['Pre'].max(), df['Post'].max())
    limit = data_max * 1.05 
    
    # --- 3. PLOTTING (Micro-scaled for 30mm) ---
    # s=1.0 keeps dots crisp at 30mm
    ax.scatter(df_stable['Pre'], df_stable['Post'], 
               c=palette['Stable'], s=1.0, alpha=1, linewidth=0, zorder=1, rasterized=True)
               
    ax.scatter(df_inhib['Pre'], df_inhib['Post'], 
               c=palette['Inhibited'], s=1.0, alpha=0.85, linewidth=0, zorder=2, rasterized=True)
               
    ax.scatter(df_excit['Pre'], df_excit['Post'], 
               c=palette['Excited'], s=1.0, alpha=0.85, linewidth=0, zorder=3, rasterized=True)

    # --- 4. THRESHOLD LINES (Micro-scaled) ---
    ax.plot([0, limit], [0, limit], color='#333333', linestyle='--', lw=0.5, zorder=0)
    # Cones drop to hair-thin 0.3
    ax.plot([0, limit], [0, limit * threshold_up], color='gray', linestyle=':', lw=0.3, zorder=0)
    ax.plot([0, limit], [0, limit * threshold_down], color='gray', linestyle=':', lw=0.3, zorder=0)

    # --- 5. MINIMALIST TEXT ANNOTATION ---
    # Fontsize reduced to 5 to avoid crushing the data points
    ax.text(0.05, 0.95, f"{props.get('Excited', 0)}%", 
            transform=ax.transAxes, color=palette['Excited'], 
            ha='left', va='top', fontsize=6)
            
    ax.text(0.95, 0.05, f"{props.get('Inhibited', 0)}%", 
            transform=ax.transAxes, color=palette['Inhibited'], 
            ha='right', va='bottom', fontsize=6)

    # --- 6. FORMATTING ---
    ax.set_aspect('equal') 
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    
    ax.set_xlabel('Pre-DCZ rate (Hz)',  labelpad=1)
    ax.set_ylabel('Post-DCZ rate (Hz)', labelpad=1)
    
    # Restrict to maximum 3 or 4 ticks so numbers don't overlap on a 30mm axis
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))  
    # Tick length stays at 2pt
    #ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 7. STRICT EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)      
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_stat_unpaired_multigroup(data_list, para_labels, group_labels, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots unpaired data (Modulation Index) across multiple parameters and animal groups.
    Utilizes a Box + Strip plot layout, ideal for small sample sizes 
    """
    set_pub_style()  
    n_params = len(data_list)
    n_hues = len(data_list[0])  # Number of animal groups (e.g., 2 or 3)
    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    edge_gray = (0, 0, 0, 0.8)  # 80% black for clean edges
    
    # --- 1. LAYOUT MATH ---
    # Determine dodge offsets dynamically based on whether there are 2 or 3 animal groups
    if n_hues == 2:
        offsets = [-0.2, 0.2]
        box_width = 0.25
    elif n_hues == 3:
        offsets = [-0.28, 0.0, 0.28]
        box_width = 0.2
    else:
        raise ValueError("This function is optimized for 2 or 3 hue groups.")

    # --- 2. DYNAMIC Y-AXIS SCALING ---
    # Safely extract all values to find absolute global bounds
    all_vals = [val for param in data_list for hue in param for val in hue if not np.isnan(val)]
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min
    
    # 25% headroom for statistical brackets
    ax.set_ylim(global_min - (y_range * 0.05), global_max + (y_range * 0.35))
    # Draw the crucial zero-baseline for Modulation Index
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.75, zorder=0, alpha=0.5)

    # --- 3. LOOP THROUGH PARAMETERS (e.g., Facilitated vs Suppressed) ---
    for p_idx, param_data in enumerate(data_list):
        metadata["Statistics"][para_labels[p_idx]] = {}
        
        # Local ceiling for bracket stacking
        local_max = max([np.nanmax(h) if len(h) > 0 else -np.inf for h in param_data])
        current_roof = local_max + (y_range * 0.08)
        
        # A. Plot Each Animal Group (Hue)
        for h_idx, hue_data in enumerate(param_data):
            clean_data = hue_data[~np.isnan(hue_data)]
            n_animals = len(clean_data)
            
            metadata["Statistics"][para_labels[p_idx]][group_labels[h_idx]] = {"N_animals": n_animals,
                                                                              'Mean values': np.mean(clean_data)}
            if n_animals == 0: continue
            
            x_pos = p_idx + offsets[h_idx]
            
            # Boxplot
            color = colors[h_idx]
            # Face is transparent (0.4), edges and whiskers are solid (1.0)
            face_color_rgba = mcolors.to_rgba(color, alpha=0.4)               
            # Pure Matplotlib Boxplot
            ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))               
            # Pure Matplotlib Scatter (s=2.5
            jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)            
            #x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=n_mice)
            ax.scatter(jittered_x, clean_data, s=2.5, color=color, alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # B. Unpaired Statistical Brackets (Comparisons against Group 0)
        # Check Group 0 vs Group 1
        if len(param_data[0]) > 2 and len(param_data[1]) > 2:
            #use ttest=0 (Mann-Whitney) and paired=0 for independent animals
            current_roof = add_stat_annotation_two_sided(
                ax, param_data[0], param_data[1], 
                p_idx + offsets[0], p_idx + offsets[1], 
                current_roof, ttest=0, paired=0)
            
            # Log exact raw p-value to metadata
            _, p_mwu = stats.mannwhitneyu(param_data[0], param_data[1], alternative='two-sided')
            metadata["Statistics"][para_labels[p_idx]][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)

        # Check Group 0 vs Group 2 (If 3 groups exist)
        if n_hues == 3 and len(param_data[0]) > 2 and len(param_data[2]) > 2:
            # Stack the bracket higher using the returned roof + a small margin
            current_roof += (y_range * 0.08)
            add_stat_annotation_two_sided(
                ax, param_data[0], param_data[2], 
                p_idx + offsets[0], p_idx + offsets[2], 
                current_roof, ttest=0, paired=0)
            
            # Log exact raw p-value to metadata
            # NOTE: If evaluating 3 groups for final publication, a Kruskal-Wallis omnibus test 
            # with Bonferroni correction is statistically mandated before Mann-Whitney.
            _, p_mwu = stats.mannwhitneyu(param_data[0], param_data[2], alternative='two-sided')
            metadata["Statistics"][para_labels[p_idx]][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

    # --- 4. FORMATTING & AESTHETICS ---
    ax.set_xticks(range(n_params))
    ax.set_xticklabels(para_labels, fontsize=6)
    ax.set_ylabel(y_label, labelpad=1)
    
    # 5 ticks max handles symmetric negative/positive scaling well
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    #ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    # --- 6. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 4 Novel context exploration before conditioning of EC3 cells--statistics

In [ ]:
def extract_data_among_group(df, para_key, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]][para_key].values)
    return data_group

group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_09_2_baseline_stat' 

ds_all_group_stat =[]
dict_event_rate = {}
dict_iei = {}

ds_all_group_dis = [] # Running distance
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    # Over all statistics
    df_stat = pd.read_csv(os.path.join(dpath_test, group_name[i]+"_statistics_with_spikes.csv")) 
    ds_all_group_stat.append(df_stat)

    # Event rate in cell-wise
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "Baseline_event_rate_with_spikes.csv")) 
    dict_event_rate[group_keys[i]] = [group['Event_rate(Hz)'].to_numpy() for _, group in df_event_rate.groupby('Animal')]
    
    df_iei = pd.read_csv(os.path.join(dpath_test, "Baseline_iei_with_spikes.csv")) 
    dict_iei[group_keys[i]] = [group['IEI(s)'].to_numpy() for _, group in df_iei.groupby('Animal')]

ds_all_group_stat = pd.concat(ds_all_group_stat, keys=group_keys ,names=["group", "row"])

# 1. CDF plot + box plot
height_mm = 40 #  
width_mm = 40 #
plot_cdf_cell_wise(dict_event_rate, group_keys, 'Event rate (Hz)', width_mm, height_mm, colors_beh_i, dpath_plot, '04_1_1_Baseline_Event_rate_cdf_plot-EC3')
plot_cdf_cell_wise(dict_iei, group_keys, 'Inter-event interval (s)', width_mm, height_mm, colors_beh_i, dpath_plot, '04_2_1_Baseline_IEI_cdf_plot-EC3')
height_mm = 20 #  
width_mm = 15 #
list_data = extract_data_among_group(ds_all_group_stat, 'Event_rate_mean(Hz)', group_keys, group_size)
plot_boxplot_horizontal(list_data, group_keys, 'Event rate (Hz)', width_mm, height_mm, colors_beh_i, dpath_plot, '04_1_2_Baseline_Event_rate_box_plot-EC3')

list_data = extract_data_among_group(ds_all_group_stat, 'IEI_mean(s)', group_keys, group_size)
plot_boxplot_horizontal(list_data, group_keys, 'Inter-event interval (s)', width_mm, height_mm, colors_beh_i, dpath_plot, '04_2_2_Baseline_IEI_box_plot-EC3')
# 2. Statitical plot--meta data
height_mm = 25 #  
width_mm = 20 #
xlabel = 'Explor' # Speed > 2cm/s
list_data = extract_data_among_group(ds_all_group_stat, 'Prop_running(%)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Running proportion (%)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '04_3_Baseline_running_proportion-EC3')

list_data = extract_data_among_group(ds_all_group_stat, 'Speed_mean(cm/s)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Average speed (cm/s)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '04_4_Baseline_Average Speed-EC3')

height_mm = 30 #  
width_mm = 20 #
# 3. Statitical plot--
list_data = extract_data_among_group(ds_all_group_stat, 'Coactivity_Prob(200ms)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Co-activity probability', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '04_5_Baseline_Co-activity probability(200ms)-EC3')

list_data = extract_data_among_group(ds_all_group_stat, 'Ensemble_Sparseness(%)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Active cells (%)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '04_6_Baseline_Ensemble_Sparseness-EC3')

# 3 Stat_supplementary
list_data = extract_data_among_group(ds_all_group_stat, 'Event_rate_max(Hz)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Max event rate (Hz)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_04_1_Baseline_Max Event Rate (Hz)-EC3')

list_data = extract_data_among_group(ds_all_group_stat, 'Skewness', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Skewness (a.u.)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_04_2_Baseline_Skewness-EC3')

list_data = extract_data_among_group(ds_all_group_stat, 'Burst_width_mean(s)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Burst width (s)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_04_3_Baseline_Burst_width_mean(s)-EC3')
print('All finished************') 

In [ ]:
def plot_stat_unpaired_single_metric(data_list, group_labels, y_label, xlabel, width_mm, height_mm, colors, output_path, title):
    """
    Plots unpaired positive data for a single metric across 2 or 3 animal groups.
    Utilizes a Box + Strip plot layout, ideal for small sample sizes.
    Optimized for a highly compact canvas (e.g., 25mm width x 30mm height).
    """
    set_pub_style()     
    n_groups = len(data_list)
    if n_groups not in [2, 3]:
        raise ValueError("This function is optimized for 2 or 3 animal groups.")
        
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    box_width = 0.4  # Slightly wider box since they are no longer clustered    
    # --- 1. DYNAMIC Y-AXIS SCALING ---
    # Safely extract all values to find absolute global bounds
    all_vals = [val for grp in data_list for val in grp if not np.isnan(val)]
    if len(all_vals) == 0:
        raise ValueError("No valid data found to plot.")
        
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min
    
    # Bottom bounded near minimum (or 0), top gets 35% headroom for statistical brackets
    ax.set_ylim(max(0, global_min - (y_range * 0.05)), global_max + (y_range * 0.35))
    # --- 2. LOOP THROUGH GROUPS ---
    # Local ceiling to start stacking brackets
    current_roof = global_max + (y_range * 0.08)
    
    for i, grp_data in enumerate(data_list):
        clean_data = grp_data[~np.isnan(grp_data)]
        n_animals = len(clean_data)
        
        metadata["Statistics"][group_labels[i]] = {"N_animals": n_animals}
        if n_animals == 0: continue
        
        x_pos = i
        color = colors[i]
        
        # Face is transparent (0.4), edges and whiskers are solid
        face_color_rgba = mcolors.to_rgba(color, alpha=0.4)                
        
        # Pure Matplotlib Boxplot
        ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                   patch_artist=True, showfliers=False, zorder=1,
                   boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                   medianprops=dict(color=color, linewidth=1.0),
                   whiskerprops=dict(color=color, linewidth=0.5),
                   capprops=dict(color=color, linewidth=0.5))                
                   
        # Pure Matplotlib Scatter (s=2.5 with white borders)
        jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)            
        ax.scatter(jittered_x, clean_data, s=4, color=color, alpha=1.0, 
                   edgecolors='white', linewidth=0.25, zorder=2)

    # --- 3. UNPAIRED STATISTICAL BRACKETS ---
    #compare against Group 0 (Control)
    if len(data_list[0]) > 2 and len(data_list[1]) > 2:
        # Group 0 vs Group 1
        current_roof = add_stat_annotation_two_sided(
            ax, data_list[0], data_list[1], 
            0, 1, current_roof, ttest=0, paired=0)
        
        _, p_mwu = stats.mannwhitneyu(data_list[0], data_list[1], alternative='two-sided')
        metadata["Statistics"][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)

    if n_groups == 3 and len(data_list[0]) > 2 and len(data_list[2]) > 2:
        # Group 0 vs Group 2 (Stack bracket slightly higher)
        current_roof += (y_range * 0.08)
        add_stat_annotation_two_sided(
            ax, data_list[0], data_list[2], 
            0, 2, current_roof, ttest=0, paired=0)
        
        _, p_mwu = stats.mannwhitneyu(data_list[0], data_list[2], alternative='two-sided')
        metadata["Statistics"][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

    # --- 4. FORMATTING & AESTHETICS ---
    #ax.set_xticks(range(n_groups))
    #ax.set_xticklabels(group_labels)
    ax.set_xticks([]) 
    ax.set_xticklabels([]) 
    ax.set_xlabel(xlabel, labelpad=2)
    ax.set_ylabel(y_label, labelpad=1)
    if (y_label == 'Running proportion (%)') | (y_label=='Average speed (cm/s)'):
         ax.set_ylabel(y_label, labelpad=1, fontsize=6)
    
    # 5 ticks max handles symmetric scaling well
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)           
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()   
    save_metadata_json(metadata, output_path, title)

In [ ]:
# 2. Custom Minor Tick Formatter
def minor_log_formatter(x, pos):
    # Extract the leading coefficient (e.g., 0.05 becomes 5)
    power = np.floor(np.log10(x))
    coeff = np.round(x / (10 ** power))    
    # Only label the '2' and '5' minor ticks to show direction clearly without overlapping
    if coeff in [ 5]:
        # Format cleanly to avoid ugly floats (e.g., prevents "0.200000001")
        if x >= 1:
            return f"{int(x)}"
        else:
            return f"{x:g}"
    return ""

def plot_boxplot_horizontal(list_data, group_labels, x_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a highly compact (e.g., 25x25mm) standalone horizontal boxplot of ANIMAL MEANS.
    Group order is visually flipped (Group 0 on top, Group N on bottom).
    Y-axis text and ticks are completely disabled for maximum conciseness.
    Input data is a list of groups, where each group is a list of animal arrays.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Initialize JSON metadata with specific keys for counts and p-values
    metadata = {
        "Figure_Title": title, 
        "Statistics": {
            "N_animals": {}, 
            "MWU_AnimalWise": {}} }
    
    edge_gray = (0, 0, 0, 0.8)
    animal_means_dict = {}
    all_means_flat = []
    
    # --- 1. EXTRACT ANIMAL MEANS SAFELY FROM LIST ---
    for i, grp in enumerate(group_labels):
        grp_means = []
        # Access the specific group's data from list_data using index 'i'
        for anim in list_data[i]:
            # Flatten array and mask out NaNs
            anim_flat = np.array(anim).flatten()
            clean_anim = anim_flat[~np.isnan(anim_flat)]
            if len(clean_anim) > 0:
                grp_means.append(np.mean(clean_anim))
                
        animal_means_dict[grp] = np.array(grp_means)
        all_means_flat.extend(grp_means)
        
        # Log the number of valid animals for this group to the JSON dictionary
        metadata["Statistics"]["N_animals"][grp] = len(grp_means)

    if len(all_means_flat) == 0:
        raise ValueError("No positive animal means found.")
        
    global_max = max(all_means_flat)
    global_min = min(all_means_flat)

    # --- 2. PLOT HORIZONTAL BOX & DOTS (Reversed Y-Order) ---
    n_groups = len(group_labels)    
    for i, grp in enumerate(group_labels):
        grp_means = animal_means_dict[grp]
        if len(grp_means) == 0: continue
        
        # Reverse the Y-position: index 0 (EC3-C) goes to the top, index 1 goes to bottom
        y_pos = n_groups - 1 - i         
        box_rgba = mcolors.to_rgba(colors[i], alpha=0.4)        
        box_kwargs = dict(patch_artist=True, showfliers=False, zorder=1,
                          medianprops=dict(color=edge_gray, linewidth=0.75),
                          whiskerprops=dict(color=edge_gray, linewidth=0.5),
                          capprops=dict(color=edge_gray, linewidth=0.5))
        
        # vert=False flips the box horizontally
        ax.boxplot(grp_means, positions=[y_pos], widths=0.4, vert=False,
                   boxprops=dict(facecolor=box_rgba, edgecolor=edge_gray, linewidth=0.5), **box_kwargs)
        
        # Jitter the animal dots vertically around their specific y_pos
        jittered_y = y_pos + np.random.uniform(-0.1, 0.1, size=len(grp_means))
        ax.scatter(grp_means, jittered_y, s=4.0, color=colors[i], edgecolors='white', linewidth=0.5, zorder=2)

    # --- 3. RIGOROUS STATISTICS (Horizontal Log Brackets) ---
    if n_groups > 1:
        ctrl_means = animal_means_dict[group_labels[0]]
        y_pos_ctrl = n_groups - 1  # Control is at the top
        
        # Calculate visual gaps for the horizontal log X-axis
        log_range = np.log10(global_max) - np.log10(global_min)
        line_gap_factor = 10 ** (log_range * 0.2)
        text_offset_factor = 10 ** (log_range * 0.3)
        
        # The 'roof' is now the furthest point on the right (X-axis)
        current_roof = np.max([np.max(animal_means_dict[g]) for g in group_labels if len(animal_means_dict[g]) > 0])
        n_comparisons = n_groups - 1
        
        for i in range(1, n_groups):
            comp_means = animal_means_dict[group_labels[i]]
            y_pos_comp = n_groups - 1 - i  # Comparison group position
            
            if len(ctrl_means) > 2 and len(comp_means) > 2:
                _, p_val = stats.mannwhitneyu(ctrl_means, comp_means, alternative='two-sided')
                p_val_adj = min(p_val * n_comparisons, 1.0) 
                
                # Log exact adjusted p-value to metadata
                metadata["Statistics"]["MWU_AnimalWise"][f"{group_labels[0]}_vs_{group_labels[i]}"] = float(p_val_adj)
                metadata["Statistics"]["Mean_values"] = {'ctr': np.mean(ctrl_means),
                                                          'comp': np.mean(comp_means)}
                star = get_asterisks(p_val_adj)
                if star and star != 'ns':
                    # Shift horizontally to the right
                    x_line = current_roof * line_gap_factor
                    x_text = x_line * text_offset_factor
                    
                    # Draw vertical bracket connecting the two Y positions
                    ax.plot([x_line, x_line], [y_pos_ctrl, y_pos_comp], lw=0.5, c='k', zorder=3)                    
                    # Place star just to the right of the bracket, centered vertically
                    ax.text(x_text, (y_pos_ctrl + y_pos_comp) / 2, star, ha='left', va='center', color='k', fontsize=7, zorder=3)
                    
                    # Push the roof further right for subsequent brackets
                    current_roof = x_text * line_gap_factor 
                    
        ax.set_xlim(global_min * 0.6, current_roof * line_gap_factor)
    else:
        ax.set_xlim(global_min * 0.6, global_max * 1.5)
        
    # Frame the Y-axis cleanly around custom positions
    ax.set_ylim(-0.5, n_groups - 0.5)

    # --- 4. FORMATTING ---
    ax.set_xscale('log')   
    # 1. Major Ticks (The Decades)
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=4))
    ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    ax.tick_params(axis='x', which='major', length=2.5, width=0.75, pad=1, labelsize=5)
    
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(ticker.FuncFormatter(minor_log_formatter))   
    ax.tick_params(axis='x', which='minor', length=1.0, width=0.3, pad=1, labelsize=5)    
    
    # Completely disable Y-axis labels and ticks for conciseness
    ax.set_yticks([]) 
    ax.set_yticklabels([])   
    
    # Intentionally removing X and Y axis text labels
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)            
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300, transparent=True)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_cdf_cell_wise(data_dict, group_labels, x_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a log-scaled CDF (e.g., 40x40mm) showing whole population distribution.
    Uses robust NumPy concatenation and a 0.001 threshold to center the data band.
    No significance testing included.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Cell_Counts": {}}    
    # --- 1. ROBUST DATA EXTRACTION & THRESHOLDING ---
    clean_cells_dict = {}
    global_max = 0.01 # Fallback max
    
    for grp in group_labels:
        # Safely flatten and concatenate all animal arrays into one 1D array
        grp_all_cells = np.concatenate([np.array(anim).flatten() for anim in data_dict[grp]])       
        # Filter NaNs and apply the 0.001 threshold to center the log scale
        clean_cells = grp_all_cells[(~np.isnan(grp_all_cells)) & (grp_all_cells >= 1e-3)]
        clean_cells_dict[grp] = clean_cells
        metadata["Cell_Counts"][grp] = len(clean_cells)
        
        if len(clean_cells) > 0 and clean_cells.max() > global_max:
            global_max = clean_cells.max()
    # --- 2. PLOT EXACT CDF ---
    for i, grp in enumerate(group_labels):
        cells = clean_cells_dict[grp]
        if len(cells) == 0: continue
        
        x_sorted = np.sort(cells)
        y_cdf = np.arange(1, len(x_sorted) + 1) / len(x_sorted)
        
        ax.plot(x_sorted, y_cdf, color=colors[i], lw=1.0, zorder=3)
        ax.scatter(x_sorted, y_cdf, s=0.2, color=colors[i], alpha=0.1, linewidth=0, zorder=1, rasterized=True)

    # --- 3. OPTIMIZED LOG FORMATTING & CENTERING ---
    ax.set_xscale('log')   
    # Explicitly bound the X-axis to lock the data into the center
    if x_label == 'Event rate (Hz)':
        ax.set_xlim(left=1e-3, right=global_max * 1.5)
    else:
        ax.set_xlim(left=0.5, right=global_max * 1.5)
    
    ax.set_xlabel(x_label,  labelpad=1)
    ax.set_ylabel('Cumulative fraction', labelpad=1)
    ax.set_ylim(0, 1.05)
    
    # Log X-Axis Ticks
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=5))
    ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    ax.tick_params(axis='x', which='major', length=3.0, width=0.75, pad=1, labelsize=6)
    
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.tick_params(axis='x', which='minor', length=1.5, width=0.3)

    # Linear Y-Axis Ticks
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax.tick_params(axis='y', length=2, pad=1)
    
    # Structural Spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Clean Legend
    custom_lines = [plt.Line2D([0], [0], color=colors[i], lw=1.5) for i in range(len(group_labels))]
    ax.legend(custom_lines, group_labels, frameon=False, loc='upper left', fontsize=5, handlelength=1.5, handletextpad=0.4)
    
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)           
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 5 Novel context exploration before conditioning of CA1 cells--statistics

In [ ]:
#For CA1
def extract_data_among_group(df, para_key, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]][para_key].values)
    return data_group

group_name = ['02.CA1-C', '03.CA1-I']
group_keys = ['CA1-C', 'CA1-I'] 
group_size = len(group_name)

test_algori_data = 'post_09_2_baseline_stat' 

ds_all_group_stat =[]
dict_event_rate = {}
dict_iei = {}

ds_all_group_dis = [] # Running distance
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    # Over all statistics
    df_stat = pd.read_csv(os.path.join(dpath_test, group_name[i]+"_statistics_with_spikes.csv")) 
    ds_all_group_stat.append(df_stat)

    # Event rate in cell-wise
    df_event_rate = pd.read_csv(os.path.join(dpath_test, "Baseline_event_rate_with_spikes.csv")) 
    dict_event_rate[group_keys[i]] = [group['Event_rate(Hz)'].to_numpy() for _, group in df_event_rate.groupby('Animal')]
    
    df_iei = pd.read_csv(os.path.join(dpath_test, "Baseline_iei_with_spikes.csv")) 
    dict_iei[group_keys[i]] = [group['IEI(s)'].to_numpy() for _, group in df_iei.groupby('Animal')]

ds_all_group_stat = pd.concat(ds_all_group_stat, keys=group_keys ,names=["group", "row"])

# 1. CDF plot + box plot
height_mm = 40 #  
width_mm = 40 #
plot_cdf_cell_wise(dict_event_rate, group_keys, 'Event rate (Hz)', width_mm, height_mm, colors_beh_i, dpath_plot, '05_1_1_Baseline_Event_rate_cdf_plot-CA1')
plot_cdf_cell_wise(dict_iei, group_keys, 'Inter-event interval (s)', width_mm, height_mm, colors_beh_i, dpath_plot, '05_2_1_Baseline_IEI_cdf_plot-CA1')
height_mm = 20 #  
width_mm = 15 #
list_data = extract_data_among_group(ds_all_group_stat, 'Event_rate_mean(Hz)', group_keys, group_size)
plot_boxplot_horizontal(list_data, group_keys, 'Event rate (Hz)', width_mm, height_mm, colors_beh_i, dpath_plot, '05_1_2_Baseline_Event_rate_box_plot-CA1')

list_data = extract_data_among_group(ds_all_group_stat, 'IEI_mean(s)', group_keys, group_size)
plot_boxplot_horizontal(list_data, group_keys, 'Inter-event interval (s)', width_mm, height_mm, colors_beh_i, dpath_plot, '05_2_2_Baseline_IEI_box_plot-CA1')

# 2. Statitical plot--meta data
height_mm = 25 #  
width_mm = 20 #
xlabel = 'Explor' # Speed > cm/s
list_data = extract_data_among_group(ds_all_group_stat, 'Prop_running(%)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Running proportion (%)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '05_3_Baseline_running_proportion-CA1')

list_data = extract_data_among_group(ds_all_group_stat, 'Speed_mean(cm/s)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Average speed (cm/s)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '05_4_Baseline_Average Speed-CA1')

height_mm = 30 # 
# 3. Statitical plot--
list_data = extract_data_among_group(ds_all_group_stat, 'Coactivity_Prob(200ms)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Co-activity probability', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '05_5_Baseline_Co-activity probability(200ms)-CA1')

list_data = extract_data_among_group(ds_all_group_stat, 'Ensemble_Sparseness(%)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Active cells (%)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, '05_6_Baseline_Ensemble_Sparseness-CA1')

# 3 Stat_supplementary
list_data = extract_data_among_group(ds_all_group_stat, 'Event_rate_max(Hz)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Max event rate (Hz)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_05_2_Baseline_Max Event Rate (Hz)-CA1')

list_data = extract_data_among_group(ds_all_group_stat, 'Skewness', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Skewness (a.u.)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_05_2_Baseline_Skewness-CA1')

list_data = extract_data_among_group(ds_all_group_stat, 'Burst_width_mean(s)', group_keys, group_size)
plot_stat_unpaired_single_metric(list_data, group_keys, 'Burst width (s)', xlabel, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_05_3_Baseline_Burst_width_mean(s)-CA1')

print('All finished************') 

In [ ]:
def plot_stat_unpaired_single_metric(data_list, group_labels, y_label, xlabel, width_mm, height_mm, colors, output_path, title):
    """
    Plots unpaired positive data for a single metric across 2 or 3 animal groups.
    Utilizes a Box + Strip plot layout, ideal for small sample sizes.
    Optimized for a highly compact canvas
    """
    set_pub_style()     
    n_groups = len(data_list)
    if n_groups not in [2, 3]:
        raise ValueError("This function is optimized for 2 or 3 animal groups.")
        
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    box_width = 0.4  # Slightly wider box since they are no longer clustered    
    # --- 1. DYNAMIC Y-AXIS SCALING ---
    # Safely extract all values to find absolute global bounds
    all_vals = [val for grp in data_list for val in grp if not np.isnan(val)]
    if len(all_vals) == 0:
        raise ValueError("No valid data found to plot.")
        
    global_max = max(all_vals)
    global_min = min(all_vals)
    y_range = global_max - global_min
    
    # Bottom bounded near minimum (or 0), top gets 35% headroom for statistical brackets
    ax.set_ylim(max(0, global_min - (y_range * 0.05)), global_max + (y_range * 0.35))
    # --- 2. LOOP THROUGH GROUPS ---
    # Local ceiling to start stacking brackets
    current_roof = global_max + (y_range * 0.08)
    
    for i, grp_data in enumerate(data_list):
        clean_data = grp_data[~np.isnan(grp_data)]
        n_animals = len(clean_data)
        
        metadata["Statistics"][group_labels[i]] = {"N_animals": n_animals}
        if n_animals == 0: continue
        
        x_pos = i
        color = colors[i]
        
        # Face is transparent (0.4), edges and whiskers are solid
        face_color_rgba = mcolors.to_rgba(color, alpha=0.4)                
        
        # Pure Matplotlib Boxplot
        ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                   patch_artist=True, showfliers=False, zorder=1,
                   boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                   medianprops=dict(color=color, linewidth=1.0),
                   whiskerprops=dict(color=color, linewidth=0.5),
                   capprops=dict(color=color, linewidth=0.5))                
                   
        # Pure Matplotlib Scatter (s=2.5 with white borders)
        jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)            
        ax.scatter(jittered_x, clean_data, s=4, color=color, alpha=1.0, 
                   edgecolors='white', linewidth=0.25, zorder=2)

    # --- 3. UNPAIRED STATISTICAL BRACKETS ---
    # compare against Group 0 (Control)
    if len(data_list[0]) > 2 and len(data_list[1]) > 2:
        # Group 0 vs Group 1
        current_roof = add_stat_annotation_two_sided(
            ax, data_list[0], data_list[1], 
            0, 1, current_roof, ttest=0, paired=0)
        
        _, p_mwu = stats.mannwhitneyu(data_list[0], data_list[1], alternative='two-sided')
        metadata["Statistics"][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)

    if n_groups == 3 and len(data_list[0]) > 2 and len(data_list[2]) > 2:
        # Group 0 vs Group 2 (Stack bracket slightly higher)
        current_roof += (y_range * 0.08)
        add_stat_annotation_two_sided(
            ax, data_list[0], data_list[2], 
            0, 2, current_roof, ttest=0, paired=0)
        
        _, p_mwu = stats.mannwhitneyu(data_list[0], data_list[2], alternative='two-sided')
        metadata["Statistics"][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

    # --- 4. FORMATTING & AESTHETICS ---
    #ax.set_xticks(range(n_groups))
    #ax.set_xticklabels(group_labels)
    ax.set_xticks([]) 
    ax.set_xticklabels([]) 
    ax.set_xlabel(xlabel, labelpad=2)
    ax.set_ylabel(y_label, labelpad=1)
    if (y_label == 'Running proportion (%)') | (y_label=='Average speed (cm/s)'):
         ax.set_ylabel(y_label, labelpad=1, fontsize=6)
    # 5 ticks max handles symmetric scaling well
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)           
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()   
    save_metadata_json(metadata, output_path, title)

In [ ]:
# 2. Custom Minor Tick Formatter
def minor_log_formatter(x, pos):
    # Extract the leading coefficient (e.g., 0.05 becomes 5)
    power = np.floor(np.log10(x))
    coeff = np.round(x / (10 ** power))    
    # Only label the '2' and '5' minor ticks to show direction clearly without overlapping
    if coeff in [ 5]:
        # Format cleanly to avoid ugly floats (e.g., prevents "0.200000001")
        if x >= 1:
            return f"{int(x)}"
        else:
            return f"{x:g}"
    return ""

def plot_boxplot_horizontal(list_data, group_labels, x_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a highly compact (e.g., 25x25mm) standalone horizontal boxplot of ANIMAL MEANS.
    Group order is visually flipped (Group 0 on top, Group N on bottom).
    Y-axis text and ticks are completely disabled for maximum conciseness.
    Input data is a list of groups, where each group is a list of animal arrays.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Initialize JSON metadata with specific keys for counts and p-values
    metadata = {
        "Figure_Title": title, 
        "Statistics": {
            "N_animals": {}, 
            "MWU_AnimalWise": {}} }
    
    edge_gray = (0, 0, 0, 0.8)
    animal_means_dict = {}
    all_means_flat = []
    
    # --- 1. EXTRACT ANIMAL MEANS SAFELY FROM LIST ---
    for i, grp in enumerate(group_labels):
        grp_means = []
        # Access the specific group's data from list_data using index 'i'
        for anim in list_data[i]:
            # Flatten array and mask out NaNs
            anim_flat = np.array(anim).flatten()
            clean_anim = anim_flat[~np.isnan(anim_flat)]
            if len(clean_anim) > 0:
                grp_means.append(np.mean(clean_anim))
                
        animal_means_dict[grp] = np.array(grp_means)
        all_means_flat.extend(grp_means)
        
        # Log the number of valid animals for this group to the JSON dictionary
        metadata["Statistics"]["N_animals"][grp] = len(grp_means)

    if len(all_means_flat) == 0:
        raise ValueError("No positive animal means found.")
        
    global_max = max(all_means_flat)
    global_min = min(all_means_flat)

    # --- 2. PLOT HORIZONTAL BOX & DOTS (Reversed Y-Order) ---
    n_groups = len(group_labels)    
    for i, grp in enumerate(group_labels):
        grp_means = animal_means_dict[grp]
        if len(grp_means) == 0: continue
        
        # Reverse the Y-position: index 0 (EC3-C) goes to the top, index 1 goes to bottom
        y_pos = n_groups - 1 - i         
        box_rgba = mcolors.to_rgba(colors[i], alpha=0.4)        
        box_kwargs = dict(patch_artist=True, showfliers=False, zorder=1,
                          medianprops=dict(color=edge_gray, linewidth=0.75),
                          whiskerprops=dict(color=edge_gray, linewidth=0.5),
                          capprops=dict(color=edge_gray, linewidth=0.5))
        
        # vert=False flips the box horizontally
        ax.boxplot(grp_means, positions=[y_pos], widths=0.4, vert=False,
                   boxprops=dict(facecolor=box_rgba, edgecolor=edge_gray, linewidth=0.5), **box_kwargs)
        
        # Jitter the animal dots vertically around their specific y_pos
        jittered_y = y_pos + np.random.uniform(-0.1, 0.1, size=len(grp_means))
        ax.scatter(grp_means, jittered_y, s=4.0, color=colors[i], edgecolors='white', linewidth=0.5, zorder=2)

    # --- 3. RIGOROUS STATISTICS (Horizontal Log Brackets) ---
    if n_groups > 1:
        ctrl_means = animal_means_dict[group_labels[0]]
        y_pos_ctrl = n_groups - 1  # Control is at the top
        
        # Calculate visual gaps for the horizontal log X-axis
        log_range = np.log10(global_max) - np.log10(global_min)
        line_gap_factor = 10 ** (log_range * 0.2)
        text_offset_factor = 10 ** (log_range * 0.3)
        
        # The 'roof' is now the furthest point on the right (X-axis)
        current_roof = np.max([np.max(animal_means_dict[g]) for g in group_labels if len(animal_means_dict[g]) > 0])
        n_comparisons = n_groups - 1
        
        for i in range(1, n_groups):
            comp_means = animal_means_dict[group_labels[i]]
            y_pos_comp = n_groups - 1 - i  # Comparison group position
            
            if len(ctrl_means) > 2 and len(comp_means) > 2:
                _, p_val = stats.mannwhitneyu(ctrl_means, comp_means, alternative='two-sided')
                p_val_adj = min(p_val * n_comparisons, 1.0) 
                
                # Log exact adjusted p-value to metadata
                metadata["Statistics"]["MWU_AnimalWise"][f"{group_labels[0]}_vs_{group_labels[i]}"] = float(p_val_adj)
                metadata["Statistics"]["Mean_values"] = {'ctr': np.mean(ctrl_means),
                                                          'comp': np.mean(comp_means)}
                star = get_asterisks(p_val_adj)
                if star and star != 'ns':
                    # Shift horizontally to the right
                    x_line = current_roof * line_gap_factor
                    x_text = x_line * text_offset_factor
                    
                    # Draw vertical bracket connecting the two Y positions
                    ax.plot([x_line, x_line], [y_pos_ctrl, y_pos_comp], lw=0.5, c='k', zorder=3)                    
                    # Place star just to the right of the bracket, centered vertically
                    ax.text(x_text, (y_pos_ctrl + y_pos_comp) / 2, star, ha='left', va='center', color='k', fontsize=7, zorder=3)
                    
                    # Push the roof further right for subsequent brackets
                    current_roof = x_text * line_gap_factor 
                    
        ax.set_xlim(global_min * 0.6, current_roof * line_gap_factor)
    else:
        ax.set_xlim(global_min * 0.6, global_max * 1.5)
        
    # Frame the Y-axis
    ax.set_ylim(-0.5, n_groups - 0.5)

    # --- 4. FORMATTING ---
    ax.set_xscale('log')   
    # 1. Major Ticks (The Decades)
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=4))
    ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    ax.tick_params(axis='x', which='major', length=2.5, width=0.75, pad=1, labelsize=5)
    
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(ticker.FuncFormatter(minor_log_formatter))   
    ax.tick_params(axis='x', which='minor', length=1.0, width=0.3, pad=1, labelsize=5)    
    
    # Completely disable Y-axis labels and ticks for conciseness
    ax.set_yticks([]) 
    ax.set_yticklabels([])   
    
    # Intentionally removing X and Y axis text labels
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)            
    
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=300, transparent=True)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_cdf_cell_wise(data_dict, group_labels, x_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a log-scaled CDF (e.g., 40x40mm) showing whole population distribution.
    Uses robust NumPy concatenation and a 0.001 threshold to center the data band.
    No significance testing included.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    metadata = {"Figure_Title": title, "Cell_Counts": {}}    
    # --- 1. ROBUST DATA EXTRACTION & THRESHOLDING ---
    clean_cells_dict = {}
    global_max = 0.01 # Fallback max
    
    for grp in group_labels:
        # Safely flatten and concatenate all animal arrays into one 1D array
        grp_all_cells = np.concatenate([np.array(anim).flatten() for anim in data_dict[grp]])       
        # Filter NaNs and apply the 0.001 threshold to center the log scale
        clean_cells = grp_all_cells[(~np.isnan(grp_all_cells)) & (grp_all_cells >= 1e-3)]
        clean_cells_dict[grp] = clean_cells
        metadata["Cell_Counts"][grp] = len(clean_cells)
        
        if len(clean_cells) > 0 and clean_cells.max() > global_max:
            global_max = clean_cells.max()
    # --- 2. PLOT EXACT CDF ---
    for i, grp in enumerate(group_labels):
        cells = clean_cells_dict[grp]
        if len(cells) == 0: continue
        
        x_sorted = np.sort(cells)
        y_cdf = np.arange(1, len(x_sorted) + 1) / len(x_sorted)
        
        ax.plot(x_sorted, y_cdf, color=colors[i], lw=1.0, zorder=3)
        ax.scatter(x_sorted, y_cdf, s=0.2, color=colors[i], alpha=0.1, linewidth=0, zorder=1, rasterized=True)

    # --- 3. OPTIMIZED LOG FORMATTING & CENTERING ---
    ax.set_xscale('log')   
    # Explicitly bound the X-axis to lock the data into the center
    if x_label == 'Event rate (Hz)':
        ax.set_xlim(left=1e-3, right=global_max * 1.5)
    else:
        ax.set_xlim(left=0.5, right=global_max * 1.5)
    
    ax.set_xlabel(x_label,  labelpad=1)
    ax.set_ylabel('Cumulative fraction', labelpad=1)
    ax.set_ylim(0, 1.05)
    
    # Log X-Axis Ticks
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=5))
    ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext())
    ax.tick_params(axis='x', which='major', length=3.0, width=0.75, pad=1, labelsize=6)
    
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs=np.arange(2, 10)))
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.tick_params(axis='x', which='minor', length=1.5, width=0.3)

    # Linear Y-Axis Ticks
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax.tick_params(axis='y', length=2, pad=1)
    
    # Structural Spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Clean Legend
    custom_lines = [plt.Line2D([0], [0], color=colors[i], lw=1.5) for i in range(len(group_labels))]
    ax.legend(custom_lines, group_labels, frameon=False, loc='upper left', fontsize=5, handlelength=1.5, handletextpad=0.4)
    
    base_path = os.path.join(output_path, title.replace(' ', '_')) 
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)           
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 6. Plot of Y maze behavioral results

In [ ]:
dic_plots = {'I_1mgKg':  ['Ymaze_C_1mgKg', 'Ymaze_I_1mgKg']}

df_stat = pd.read_csv(os.path.join(dir_stat, "03_1.Spa_Y_maze.csv")) 
height_mm = 30
width_mm = 30
colors = {'I_1mgKg': colors_beh_i}
#1.Spontaneous alteration
plot_behav_stat(df_stat, dic_plots, 'Spon_alternation', 'Alternation (%)', width_mm, height_mm, colors,  dpath_plot, '06_1_Y_maze_stat_alternation')
width_mm = 25
# 2. Totall entries
plot_behav_stat(df_stat, dic_plots, 'Entry_num', 'Total entries', width_mm, height_mm, colors,  dpath_plot, '06_2_Y_maze_stat_total_entries')
# 3. Runing distance
plot_behav_stat(df_stat, dic_plots, 'Distance', 'Runing dis. (m)', width_mm, height_mm, colors,  dpath_plot,  '06_3_Y_maze_stat_running_distance')
# 4. Max speed
plot_behav_stat(df_stat, dic_plots, 'Max_speed', 'Max speed (m/s)',width_mm, height_mm, colors,  dpath_plot, '06_4_Y_maze_stat_max_speed')

print('All finished')

In [ ]:
def plot_behav_stat(df_stat, dic_plots, data_col, y_label, width_mm, height_mm, colors_beh, dpath_plot, title):
    """
    Calculates the session-wide average and plots it as a grouped bar/scatter chart
    optimized for narrow widths. 
    """
    set_pub_style() 
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    
    # --- 1. PLOTTING SETUP ---
    main_groups = list(dic_plots.keys()) 
    base_x_positions = [0]               # Only 1 base position for the single pair
    bar_width = 0.3
    offsets = [-0.2, 0.2]                # Offsets for the 2 bars within the cluster
    
    metadata = {"Figure_Title": title, "Statistics": {data_col: {}}}
    
    global_y_max = 0
    bracket_tops = []
    
    # --- 2. DATA EXTRACTION & PLOTTING ---
    for b_idx, main_group in enumerate(main_groups):
        subgroups = dic_plots[main_group]
        base_x = base_x_positions[b_idx]
        colors = colors_beh[main_group]
        cluster_data = [] # Store data arrays for stats within this cluster later
        cluster_x = []    # Store exact X coordinates for stats bracket
        
        for g_idx, group in enumerate(subgroups):
            group_df = df_stat[df_stat['Group'] == group]
            animal_means = group_df[data_col].values.astype(float)
            animal_means = animal_means[~np.isnan(animal_means)] # Drop true NaNs
            
            if len(animal_means) == 0: 
                continue
                
            cluster_data.append(animal_means)
            
            x_pos = base_x + offsets[g_idx]
            cluster_x.append(x_pos)
            color = colors[g_idx]
            
            mean_val = np.mean(animal_means)
            sem_val = stats.sem(animal_means)                    
            
            # Track global max for dynamic Y-axis limits
            global_y_max = max(global_y_max, np.max(animal_means))
            
            # Record Metadata
            metadata["Statistics"][data_col][group] = {
                "N_mice": len(animal_means), "Mean": float(mean_val), "SEM": float(sem_val)}
            
            # A. Draw Bar (Face='none', Edge=color)
            ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
                   facecolor='none', edgecolor=color, linewidth=1.0, capsize=0,
                   error_kw=dict(lw=0.75, ecolor='black'), zorder=2)
            
            # B. Scatter Raw Data 
            x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=len(animal_means))
            ax.scatter(x_jitter, animal_means, color=color, edgecolor='none', 
                       s=3, zorder=3, alpha=1.0)
                        
        # --- 3. APPLY STATISTICS WITHIN CLUSTER ---
        # Test the two subgroups against each other within the current dosage group
        if len(cluster_data) == 2 and len(cluster_data[0]) >= 3 and len(cluster_data[1]) >= 3:
            local_max = max(np.max(cluster_data[0]), np.max(cluster_data[1]))
            error_max = max(np.mean(cluster_data[0]) + stats.sem(cluster_data[0]), 
                            np.mean(cluster_data[1]) + stats.sem(cluster_data[1]))
            top_y = max(local_max, error_max)
            
            # Call the robust annotation function and unpack the p-value
            bracket_top, p_val = add_stat_annotation_two_sided(
                ax, cluster_data[0], cluster_data[1], 
                cluster_x[0], cluster_x[1], y_max=top_y, ttest=0, paired=0)
            
            # Log the p-value to the JSON metadata dictionary
            metadata["Statistics"][data_col][f"{subgroups[0]}_vs_{subgroups[1]}_pval"] = float(p_val)
            
            if bracket_top is not None:
                bracket_tops.append(bracket_top)

    # --- 4. AESTHETICS & FORMATTING ---
    ax.set_xticks(base_x_positions)
    ax.set_xticklabels(['Y maze'], fontsize=7) # Set specific label for the single pair
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Y-axis scaling logic
    if data_col == 'Spon_alternation':
        ax.set_ylim(0, 100) # Lock to 100%
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
        ax.axhline(y=50, linestyle='--', linewidth=0.75, color='gray')
    else:
        # Dynamically set y_limit based on max data or highest significance bracket
        overall_max = max([global_y_max] + bracket_tops) if bracket_tops else global_y_max
        ax.set_ylim(0, overall_max * 1.15) # Leave 15% headroom for neatness
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
    
    # CRITICAL FIX FOR NARROW PLOTS: 
    # Force the X-axis limits to hug the bars tightly
    ax.set_xlim(base_x_positions[0] + offsets[0] - 0.4, base_x_positions[-1] + offsets[-1] + 0.4)     
    
    # Cell Reports Style: Hide top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # --- 5. SAVING OUTPUTS ---
    base_path = os.path.join(dpath_plot, title.replace(' ', '_'))        
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()          

    save_metadata_json(metadata, dpath_plot, title)

def add_stat_annotation_two_sided(ax, data1, data2, x1, x2, y_max, ttest=0, paired=0):
    """
    Calculates significance and draws a flat horizontal line.
    Defaults to Mann-Whitney (ttest=0, paired=0) for safer bounded-data statistics.
    """
    # 1. Choose Test
    if (ttest==1) and (paired==0): 
        # Welch's t-test (equal_var=False)
        stat, p = stats.ttest_ind(data1, data2, alternative='two-sided') #equal_var=False,
    elif (ttest==1) and (paired==1): 
        stat, p = stats.ttest_rel(data1, data2, alternative='two-sided')
    elif (ttest==0) and (paired==0): 
        stat, p = stats.mannwhitneyu(data1, data2, alternative='two-sided')
    elif (ttest==0) and (paired==1): 
        stat, p = stats.wilcoxon(data1, data2, alternative='two-sided')
    else:
        raise SystemExit("Stop right there! Check the test methods!")
        
    star = get_asterisks(p)
    top = add_significance_bar(ax, x1, x2, y_max, star, color='black')   
    
    return top, p